In [3]:
import yfinance as yf
import pandas as pd
from sklearn.linear_model import LogisticRegression
import joblib

tickers = ["AAPL","MSFT","GOOGL","AMZN","TSLA","NVDA","BRK-B","^GSPC","^IXIC","^FTSE","^DJI"]
all_data = {}

In [4]:
for ticker in tickers:
    stock = yf.Ticker(ticker)
    data = stock.history(period="10y")

    if data.empty:
        print(ticker, "has no data, skipping")
        continue

    data = data.sort_index(ascending=True)  # oldest to newest
    data = data[~data.index.duplicated(keep="first")]
    data = data[(data["Close"] > 0) & (data["Volume"] > 0)]
    data["Ticker"] = ticker
    if len(data) < 250:
        print(ticker, "doesn't have enough history, skipping")
        continue

    data["MA20"] = data["Close"].rolling(20).mean()
    data["MA50"] = data["Close"].rolling(50).mean()
    data["MA200"] = data["Close"].rolling(200).mean()

    data["Return5"] = (data["Close"] / data["Close"].shift(5)) - 1
    data["Return20"] = (data["Close"] / data["Close"].shift(20)) - 1
    data["MA20/MA50"] = data["MA20"] / data["MA50"]
    data["Close/MA200"] = data["Close"] / data["MA200"]
    data["Volume/Volume_MA20"] = data["Volume"] / data["Volume"].rolling(20).mean()

    data = data.dropna(subset=["Open","High","Low","Close","Volume","MA20", "MA50", "MA200"])
    data = data.reset_index()

    all_data[ticker] = data
combined_data = pd.concat(all_data.values(), ignore_index=True, axis=0, sort=False)
model = LogisticRegression()


joblib.dump(model, "logistic_regression_model.joblib")

['logistic_regression_model.joblib']